In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

start_date = "2025-05-01"
end_date = "2026-04-23"
snapshot_date = (
    datetime.strptime(str(end_date), "%Y-%m-%d") - timedelta(days=1)
).strftime("%Y%m%d")


In [0]:
snapshot_date

In [0]:
df_CC_Members = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == snapshot_date)
    .filter(F.col("renewal_status").isin("Active", "Active_Non_Renewing"))
    .filter(F.col("plan_id").isin('annual1'))
    .filter(F.col("renewal_datetime").between("2026-05-01", "2026-05-31"))
)

In [0]:
df_CC_Members.display()

In [0]:
from pyspark.sql import functions as F

T_all_trans = (
    T_trans
    .join(
        T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
        T_trans.LocationId == T_location.LocationID,
        "left"
    )
    .join(
        T_date.select("dateid", "datefull"),
        T_trans.VisitDateId == T_date.dateid,
        "left"
    )
    .filter(F.col("datefull").between(start_date, end_date))
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("CineClub_Discount_Type_ID").isin(2, 3, 5, 6))  #2 for concessions, 3 for gaming, 5 for member priced, 6 for free
    .groupBy("CDE_ID", "datefull", T_trans.LocationId)
    .agg(
        # Concession Revenue
        F.sum(
            F.when((F.col("CineClub_Discount_Type_ID") == 2) & 
                   (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")), F.col("Net_Revenue"))
        ).alias("Concession_Revenue"),

        # Gaming Revenue

        F.sum(
            F.when((F.col("CineClub_Discount_Type_ID") == 3) &
                (F.col("LC_Trade_Brand_Description").isin("Rec Room", "Playdium")), F.col("Net_Revenue"))
        ).alias("Gaming_Revenue"),

        # Ticket Revenue
        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID").isin(5, 6)) &
                (F.col("LC_Location_Type_Description") == "Cineplex Theatre"),
                F.col("Net_Revenue")
            )
        ).alias("Ticket_Revenue"),

        # Member tickets
        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID") == 5) & 
                (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")),
                F.col("Quantity")
            )
        ).alias("Member_Priced_Tickets"),

        #Tickets Redeemed

        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID") == 6) & 
                (F.col("LC_Location_Type_Description").isin("Cineplex Theatre")),
                F.col("Quantity")
            )
        ).alias("Free_Tickets"),

        #Visits

    )
    .withColumn(
        "Visits",
        F.when(
            (F.col("Concession_Revenue") > 0) |
            (F.col("Ticket_Revenue") > 0) |
            (F.col("Gaming_Revenue") > 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

In [0]:
pip install openpyxl

In [0]:
import pandas as pd
df_pricing_pd = pd.read_excel("/Volumes/data_exploration_hub/marketing_hub/offline_data/Circuit Pricing.xlsx")
df_pricing = spark.createDataFrame(df_pricing_pd)

# Explode Theatre_ID into individual rows
df_pricing = df_pricing.withColumn(
    "Theatre_ID_Array",
    F.split(F.col("Theatre_ID"), ",")
)
df_price_exploded = (
    df_pricing
    .withColumn("Theatre_ID_exploded", F.explode("Theatre_ID_Array"))
    .withColumn("Theatre_ID_clean", F.col("Theatre_ID_exploded").cast("int"))
).drop("Category_Group", "Theatre_ID", "Theatre_ID_Array", "Theatre_ID_exploded", 
       "Child", "Senior", "Cineplex_Tuesday", "Tuesday")

In [0]:
from pyspark.sql import functions as F

T_savings = (
    T_all_trans
    # total tickets
    .withColumn(
        "No_of_tickets",
        F.coalesce(F.col("Member_Priced_Tickets"), F.lit(0)) +
        F.coalesce(F.col("Free_Tickets"), F.lit(0))
    )
    # join price table
    .join(
        df_price_exploded,
        T_all_trans.LocationId == df_price_exploded.Theatre_ID_clean,
        "left"
    )
    # ticket savings
    .withColumn(
        "Ticket_Savings",
        F.coalesce(
            F.round(
                (
                    (F.col("General") - F.lit(9.99)) * F.col("No_of_tickets")
                ) +
                (F.col("No_of_tickets") * F.lit(1.5)),
                2
            ),
            F.lit(0.0)
        )
    )
    # concession savings (25%)
    .withColumn(
        "Total_Savings_Concessions",
        F.coalesce(
            F.round(F.col("Concession_Revenue") * 0.25, 2),
            F.lit(0.0)
        )
    )
    # gaming savings (25%)
    .withColumn(
        "Total_Savings_Gaming",
        F.coalesce(
            F.round(F.col("Gaming_Revenue") * 0.25, 2),
            F.lit(0.0)
        )
    )
)

In [0]:
from pyspark.sql import functions as F

T_user_level = (
    T_savings
    .groupBy("CDE_ID")
    .agg(
        # Revenue
        F.sum("Concession_Revenue").alias("Total_Concession_Revenue"),
        F.sum("Ticket_Revenue").alias("Total_Ticket_Revenue"),
        F.sum("Gaming_Revenue").alias("Total_Gaming_Revenue"),

        # Tickets
        F.sum("Member_Priced_Tickets").alias("Total_Member_Priced_Tickets"),
        F.sum("Free_Tickets").alias("Total_Free_Tickets"),
        F.sum("No_of_tickets").alias("Total_Tickets"),

        # Visits
        F.sum("Visits").alias("Total_Visits"),

        # Savings
        F.sum("Ticket_Savings").alias("Total_Ticket_Savings"),
        F.sum("Total_Savings_Concessions").alias("Total_Concession_Savings"),
        F.sum("Total_Savings_Gaming").alias("Total_Gaming_Savings")
    )
)

In [0]:
df_final =(
    T_user_level
    .join(
        df_CC_Members,
        df_CC_Members.cde_id == T_user_level.CDE_ID,
        "inner"
    )
    .select(
        T_user_level.CDE_ID,
        "cc_user_profile_id",
        "scene_membership_id",
        "plan_id",
        "renewal_status",
        "recognitions_left",
        "Total_Visits",
        "Total_Member_Priced_Tickets",
        "Total_Free_Tickets",
        "Total_Ticket_Revenue",
        "Total_Ticket_Savings",
        "Total_Concession_Revenue",
        "Total_Concession_Savings",
        "Total_Gaming_Revenue",
        "Total_Gaming_Savings"
).withColumnRenamed("recognitions_left", "Unused_Recognitions")
    .filter(F.col("Unused_recognitions")<100)
    .filter(F.col("Total_Visits")>0)
    .withColumn("Total_Savings", F.col("Total_Ticket_Savings") + F.col("Total_Concession_Savings") + F.col("Total_Gaming_Savings"))
)

In [0]:
df_final.display()

In [0]:
import pandas as pd

from pyspark.sql.types import StructType, StructField, StringType

pdf_names = pd.read_excel(
    "/Volumes/data_exploration_hub/marketing_hub/offline_data/Subscriptions - March 2026 .xlsx",
    usecols=[
        "customers.id",
        "customers.first_name",
        "customers.last_name",
    ],
    dtype="string",  # read everything as pandas nullable string up front
)

# Rename to drop the dots (Spark-friendly)
pdf_names = pdf_names.rename(columns={
    "customers.id": "customer_id",
    "customers.first_name": "first_name",
    "customers.last_name": "last_name",
})

# Filter
pdf_names = pdf_names[pdf_names["customer_id"] != "AzqBunUAsPVB41nKT"]

# Explicit schema — skips inference entirely
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
])

df_names = spark.createDataFrame(pdf_names, schema=schema)


In [0]:
T_email = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
df_final_2 = (
    df_final
    .join(
        df_names,
        df_final.cc_user_profile_id == df_names.customer_id,
        "inner"
    )
    .join(T_email, "CDE_ID", "inner")
    .select("CDE_ID", "cc_user_profile_id", "scene_membership_id", "first_name", "last_name", F.upper("email_address").alias("Email_ID"), "plan_id", "renewal_status", "Unused_Recognitions", "Total_Visits", "Total_Member_Priced_Tickets", "Total_Free_Tickets", F.round("Total_Savings",2).alias("Total_Savings"))
    .dropDuplicates(["CDE_ID"])
)

In [0]:
df_final_2.display()